In [3]:
with open('../input.txt','r',encoding='utf-8') as f:
    text = f.read()

In [ ]:
print(text[:1000])

In [ ]:
#tokenizer (encode and decode) -> character tokenizer

chars = sorted(list(set(text)))

stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i

# number -> char
itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch


def encode(s):
    result = []

    for ch in s:
        result.append(stoi[ch])

    return result


def decode(nums):
    text = ""

    for n in nums:
        text += itos[n]

    return text


encoded = encode("hi")
print(encoded)

decoded = decode(encoded)
print(decoded)

In [ ]:
#implement tiktoken tokenizer
#...

In [ ]:
#encoding entire dataset 

import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:1000])

In [ ]:
# split data into train and validation sets

n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [ ]:
#creating batches
torch.manual_seed(1337)
block_size = 8
batch_size = 4

def get_batch(split):
  if split == 'train':
    data = train_data
  else:
    data = val_data
  
  indexes = torch.randint(len(data) - block_size,(batch_size,))
  x = torch.stack([data[index:index+block_size] for index in indexes])
  y = torch.stack([data[index+1:index+block_size+1] for index in indexes])
  return x,y

xb, yb = get_batch("train")
print(xb.shape)
print("Inputs : ")
print(xb)
print(yb.shape)
print("Outputs : ")
print(yb)


In [ ]:
vocab_size

In [ ]:
#NN (Bigram model is a one character prediction lnaguage model)
# Bigram model learns:
# current token -> next token probabilities using an embedding lookup table.

# Input shape (B,T) becomes (B,T,C),
# where each token returns C=vocab_size next-token logits.

import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)


  def forward(self,idx,targets=None):
    logits = self.token_embedding_table(idx)

    #pytorch cross_entropy accepts in (X x y) format as we have 3d we are converting into 2d
    
    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits, loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss = self(idx)
      logits = logits[:,-1,:] #i want last one's logit
      probabilities = F.softmax(logits,dim=-1)
      idx_new = torch.multinomial(probabilities,num_samples=1) #returns a random idx
      idx = torch.cat((idx,idx_new),dim=1)
    return idx


model = BigramLanguageModel(vocab_size)
logits,loss = model(xb,yb)
print(logits.shape)
print(loss) #around 4.87 as model's initial weights are randomly generated so ln(1/65) = 4.17



In [ ]:
#testing initial random text
initial_idx = torch.tensor([[47]], dtype=torch.long)
encoded = model.generate(initial_idx,max_new_tokens=150)

decoded = decode(encoded[0].tolist())
print(decoded)



In [ ]:

#Now actual training step
#AdamW - Updates model's weights (those raw scores in embedding table) using gradients
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

In [ ]:
batch_size = 32
for step in range(10000):
  xb,yb = get_batch('train')
  logits,loss = model(xb,yb)
  #remove previous gradients
  optimizer.zero_grad(set_to_none=True)
  #create gradients
  loss.backward()
  #update weights
  optimizer.step()

print(loss.item())
# The model first makes predictions and calculates how wrong they are using a loss function. Backpropagation (`loss.backward()`) computes gradients, which are instructions telling
# each weight how it should change to reduce the error. Finally, the optimizer updates the weights using those gradients, helping the model make better predictions over time.

In [ ]:
initial_idx = torch.tensor([[46]], dtype=torch.long)
encoded = model.generate(initial_idx,max_new_tokens=500)

decoded = decode(encoded[0].tolist())
print(decoded)

In [ ]:
#mathematical trick in self attention
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)
print(x)

In [ ]:
# averaging
x_bag_of_words = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    x_previous = x[b,:t+1]
    x_bag_of_words[b,t] = torch.mean(x_previous,0)

In [ ]:

#now the trick using tril and matrix multiplication
weights = torch.tril(torch.ones(T,T))
weights = weights / weights.sum(1,keepdim=True)
x_bag_of_words = weights @ x

In [ ]:
#trick using softmax
tril = torch.tril(torch.ones(T, T))

wei = torch.zeros((T, T))

wei = wei.masked_fill(tril == 0, float('-inf'))

wei = F.softmax(wei, dim=-1)

x_bag_of_words3 = wei @ x

torch.allclose(x_bag_of_words, x_bag_of_words3)

In [ ]:
#SELF ATTENTION INTUITION

head_size = 16

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# Create linear layers to generate:
# keys   -> what information token contains
# queries -> what information token is looking for
# values -> actual information token will send
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)


k = key(x)
q = query(x)
v = value(x)

# Compute attention scores between every token pair
# (B,T,16) @ (B,16,T) -> (B,T,T)
weights = q @ k.transpose(-2, -1)

# Lower triangular matrix to block future tokens
# Prevents model from cheating by looking ahead
tril = torch.tril(torch.ones(T, T))

# Replace future-token positions with -inf
# After softmax they become 0 attention
weights = weights.masked_fill(tril == 0, float('-inf'))

# Convert attention scores into probabilities
# Each row now sums to 1
weights = F.softmax(weights, dim=-1)

# Gather contextual information from previous tokens
# using attention probabilities
# (B,T,T) @ (B,T,16) -> (B,T,16)
out = weights @ v

# Final contextualized token representations
out.shape

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

#hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_layer = 6
n_heads = 6
dropout = 0.2

torch.manual_seed(1337)

with open('input.txt','r',encoding='utf-8') as f:
    text = f.read()

#tokenizer (encode and decode) -> character tokenizer

chars = sorted(list(set(text)))
vocab_size = len(chars
                 )
stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i

# number -> char
itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch


def encode(s):
    result = []

    for ch in s:
        result.append(stoi[ch])

    return result


def decode(nums):
    text = ""

    for n in nums:
        text += itos[n]

    return text


encoded = encode("hi")
print(encoded)

decoded = decode(encoded)
print(decoded)

#encoding entire dataset 

data = torch.tensor(encode(text),dtype=torch.long)
print(data[:1000])

# split data into train and validation sets
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]


def get_batch(split):
  if split == 'train':
    data = train_data
  else:
    data = val_data
  
  indexes = torch.randint(len(data) - block_size,(batch_size,))
  x = torch.stack([data[index:index+block_size] for index in indexes])
  y = torch.stack([data[index+1:index+block_size+1] for index in indexes])
  x,y = x.to(device), y.to(device)
  return x,y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

#attention head class
class Head(nn.Module):
    #one head of self attention
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        # Compute attention scores between every token pair
        # (B,T,16) @ (B,16,T) -> (B,T,T)
        weights = q @ k.transpose(-2, -1) * C**-0.5

        # Replace future-token positions with -inf
        # After softmax they become 0 attention
        weights = weights.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1) # (B,T,T)
        weights =self.dropout(weights)
        v = self.value(x) # (B,T,head_size)


        out = weights @ v
        return out


#multi-head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    
    
#computation of the token
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )


    def forward(self, x):
        return self.net(x)

#transformer block
class Block(nn.Module):
    def __init__(self, n_embd, num_heads):
        super().__init__()
        head_size = n_embd // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

#bigram model
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.block = nn.Sequential(*[Block(n_embd, num_heads=n_heads) for _ in range(n_layer)])
        #final layer norm
        self.lm_f=nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        token_embeddings = self.token_embedding_table(idx)  # (B,T,C)
        position_embeddings = self.position_embedding_table(torch.arange(T,device=device))
        x = token_embeddings + position_embeddings  # (B,T,C)
        #block of transformer
        x = self.block(x) # (B,T,C)
        x = self.lm_f(x) # (B,T,C)
        logits = self.lm_head(x)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # crop context to block_size
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx
    
model = BigramLanguageModel()
m = model.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))